# PlagIA Empresarial · 02 — Pipeline Radiométrico (OpenCV CLAHE) y Geoespacial (Rasterio)

**Subsistema de Visión Computacional y Geoprocesamiento (`engine`)**

### Objetivos Técnicos
1. Verificar la corrección radiométrica **CLAHE en espacio LAB (OpenCV)** sobre capturas de arroz.
2. Construir y validar los `DataLoader` de **PyTorch** con aumento de datos **Albumentations** y `pin_memory` para CUDA.
3. Generar y seccionar un ortomosaico **GeoTIFF (`EPSG:4326`)** con **Rasterio** sobre el distrito de riego de El Espinal (Tolima), conservando la transformación afín GPS de cada parche.

In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().resolve()
while not (RAIZ_PROYECTO / "pyproject.toml").exists() and RAIZ_PROYECTO != RAIZ_PROYECTO.parent:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

import pandas as pd
import matplotlib.pyplot as plt

from engine.config import DIR_ARTEFACTOS, RUTA_MANIFIESTO_PARTICIONADO, TAMANO_LOTE
from engine.preprocessing import (
    cargar_y_preparar_imagen,
    construir_cargadores_datos,
    desnormalizar_tensor,
)
from engine.geospatial import (
    crear_ortomosaico_geotiff_muestra,
    seccionar_ortomosaico_en_parches,
)

## 1. Corrección Radiométrica CLAHE + Cargadores de Datos PyTorch

In [ ]:
df_particionado = pd.read_csv(RUTA_MANIFIESTO_PARTICIONADO)
cargadores = construir_cargadores_datos(df_particionado, batch_size=TAMANO_LOTE, apply_clahe=True)

lote_x, lote_y = next(iter(cargadores["train"]))
print("Dimensiones del lote NCHW:", lote_x.shape, "| tipo:", lote_x.dtype)
print("Etiquetas del lote       :", lote_y[:8].tolist())

ruta_muestra = df_particionado.iloc[0]["path"]
rgb_original = cargar_y_preparar_imagen(ruta_muestra, aplicar_clahe=False)
rgb_clahe = cargar_y_preparar_imagen(ruta_muestra, aplicar_clahe=True)

fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
ejes[0].imshow(rgb_original)
ejes[0].set_title("Captura Original")
ejes[0].axis("off")

ejes[1].imshow(rgb_clahe)
ejes[1].set_title("Corrección Radiométrica CLAHE")
ejes[1].axis("off")

ejes[2].imshow(desnormalizar_tensor(lote_x[0]))
ejes[2].set_title("Tensor Albumentations (Entrenamiento)")
ejes[2].axis("off")
plt.tight_layout()
plt.show()

## 2. Particionado de Ortomosaicos GeoTIFF con Rasterio

In [ ]:
imgs_muestra = [Path(p) for p in df_particionado["path"].head(9).tolist()]
tif_muestra = crear_ortomosaico_geotiff_muestra(
    DIR_ARTEFACTOS / "ortomosaicos" / "lote_espinal_muestra.tif",
    zone_key="EL_ESPINAL",
    source_images=imgs_muestra,
)
parches = seccionar_ortomosaico_en_parches(tif_muestra, patch_size=224)
print(f"Ortomosaico generado en: {tif_muestra}")
print(f"Total parches georreferenciados extraídos: {len(parches)}")
print("Primer parche GPS:", {k: v for k, v in parches[0].items() if k != "image_rgb"})